# Set-up

Kernel: 
- Local mac: .conda 3.8
- Roger GPU: .conda 3.8.18
    - 24564 MiB == 25 GB (24 Gibibyte) memory
- Roger CPU: base with Python 3.11.3

### ToDo
- LML hyperparameter optimization.
    - Currently I am optimising in scaled space, as minimum should be minimum. However ranges use for scaling vary so error translation between target and source domain vary. Is this a problem?
    - Should we use something else for hyperparameter optimisation?
    - Treat mean as a hyperparameter?
- Testing of pixel correction
    - Slightly improves RMSE, however makes hyperparameter optimisation via SGD much more shaky.
- Testing of scaling 
    - Highly sensitive to scaling.
    - Scale the auxiliary channel with the range of the HR target channel, to maintain signal relative signal strength and not scale that away.
- Testing of spatial-only kernel.
    - on aux
    - on input (simple spatial smoothing)
- Testing of upscaling-specific hyperparameters.
    - results indicate that pixel kernel sensitivity is variable.
- Testing of mean functions (current mean is -0.8)
    - treat mean as hyperparameter?
- Implement gradient kernel:
    - lr bed elevation gradients and aux gradients
- Visulaise correlation of errors with ice thickness.
- Visualise correlation of gradients.

### Questions:
- In super-resolution applications the training data are the low-resolution data points. If we only use this for hyperparameter optimization in LML, how do we reach good extrapolating capabilities?
- Mean function: Use Gaussian spatial kernel as mean function?
- What scaling is sensible?
    - Scaling for auxiliary spaces is similar to using relative changes: Use gradients directly.

### Notes:
- Scaling is performed only on LR inputs (after degradation), as it might be considered "cheating" if we use the target domain for scaling (which means that no predictions would fall outside the scaling range.)
- Bicubic is the strongest baseline.

In [58]:
import torch
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import datetime
import math
import sys
import torchmetrics

torch.set_printoptions(sci_mode = False)
# Memory consideration: 4 bytes per dp
torch.set_default_dtype(torch.float32)

In [2]:
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



## Load data

In [30]:
# 34 MB data
training_tensor_full = torch.load(
    './torch_data/DOMEC_bed_scenes_60pixel.pt')

training_tensor_full = torch.load(
    './torch_data/DOMEC_bed_scenes.pt')

In [54]:
# actually there is a small neg correlation
torch.corrcoef(torch.cat((training_tensor_full[36, 0, :, :].reshape(1, -1),
          training_tensor_full[36, 1, :, :].reshape(1, -1)), dim = 0))

tensor([[ 1.0000, -0.1336],
        [-0.1336,  1.0000]])

In [65]:
dx, dy = torchmetrics.functional.image.image_gradients(training_tensor_full[36, 0, :, :].unsqueeze(0).unsqueeze(0))
dxs, dys = torchmetrics.functional.image.image_gradients(training_tensor_full[36, 1, :, :].unsqueeze(0).unsqueeze(0))
dx.reshape(1, -1)

torch.corrcoef(torch.cat((dx.reshape(1, -1),
           dy.reshape(1, -1),
           dxs.reshape(1, -1),
           dys.reshape(1, -1)), dim = 0))

tensor([[ 1.0000, -0.2053,  0.9897, -0.2481],
        [-0.2053,  1.0000, -0.1961,  0.8882],
        [ 0.9897, -0.1961,  1.0000, -0.2337],
        [-0.2481,  0.8882, -0.2337,  1.0000]])

In [46]:
fig = px.imshow(training_tensor_full[36, 1, :, :].reshape(45, 45).cpu(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "RMSE distribution")
fig.show()

In [5]:
torch.corrcoef(training_tensor_full.permute(1, 0, 2, 3).reshape(8, -1))
# 0.7 correlation

tensor([[     1.0000,     -0.2104,     -0.9220,         nan,     -0.3951,
              0.0247,     -0.1630,     -0.3196],
        [    -0.2104,      1.0000,      0.5726,         nan,      0.9183,
             -0.1270,      0.4356,      0.8255],
        [    -0.9220,      0.5726,      1.0000,         nan,      0.6952,
             -0.0710,      0.3093,      0.5951],
        [        nan,         nan,         nan,         nan,         nan,
                 nan,         nan,         nan],
        [    -0.3951,      0.9183,      0.6952,         nan,      1.0000,
             -0.1948,      0.3983,      0.8600],
        [     0.0247,     -0.1270,     -0.0710,         nan,     -0.1948,
              1.0000,      0.1265,     -0.2398],
        [    -0.1630,      0.4356,      0.3093,         nan,      0.3983,
              0.1265,      1.0000,     -0.0000],
        [    -0.3196,      0.8255,      0.5951,         nan,      0.8600,
             -0.2398,     -0.0000,      1.0000]])

In [6]:
# Ice thickness in this area is mean 793
print(torch.mean(training_tensor_full[:, 2, :, :]))
print(torch.min(training_tensor_full[:, 0, :, :]))
print(torch.max(training_tensor_full[:, 0, :, :]))

tensor(3230.3552)
tensor(-1470.0149)
tensor(1116.6499)


In [7]:
# Subset first two channels
training_tensor = training_tensor_full[:, (0, 1), :, :].to(device)

## PSNR metric

For PSNR, MAX is the maximum signal. It traditionally is defined over a bounded space as the upper bound of the MSE. E.g. 255 is the upper bound for regular RGB image predictions as values of each colour channel range [0, 255]

- **Option 1: Original domain PSNR**: Use the range of bed elevation values across the domain as the Max signal. We use range instead of max() because out target also taked on negative values.
- **Option 2: Scaled PNSR**: Calulate the MSE in the transformed (scaled) input space where 20 is PSNR max, because input values were scaled between [-10, 10]. Due to different scaling transformations for each scene, this metric will thus focus more on assessment of model but less on impact of transformation.

In [8]:
mse_example = 20.0
PSNR_max_range_meters = torch.max(training_tensor[:, 0, :, :]) - torch.min(training_tensor[:, 0, :, :])

psnr = torch.mul(
            input = torch.log10(torch.div(
                torch.pow(PSNR_max_range_meters, exponent = 2), 
                mse_example)), 
            other = 10.0)

print("Example of PSNR", np.round(psnr.item(), 5))

Example of PSNR 55.2445


## Class

In [9]:
class Superresolution_GP:
    # Initialise object only over overall dimensionalitie
    def __init__(self, hr_hw, up_factors, initial_hyperparameters, device):
        """_summary_

        Args:
    
            hr_hw (torch.tensor(size = [1])): 1d torch_tensor containing the dimensionality of the target/ground truth scenes. 
                Scenes are always square with H == W. Must not be on device yet, as this will be initialised.
            up_factors (torch.tensor(size = [n_up_factors])): 1d torch tensor containing all upscaling factors (magnification factors) which will be applied.
            initial_hyperparameters(torch.tensor(size = [3, 1])): lambda_s, lambda_p, lambda_f, 
                must be .to(torch.float)
            device (device object): device(type='cuda')

        Raises:
            ValueError: Error if the upscaling_factors don't perfectly match hr_hw.
        """
        self.hr_hw = hr_hw.to(device) # High-resolution W (width) which is equal to H: H_h, W_h. torch.Size([1])
        self.up_factors = up_factors.to(device)  # torch.Size([5])
        # Extract number of up_factors
        self.n_up_factors = torch.tensor(self.up_factors.shape, device = device)
        # Low-resolution HW for each different up-factor, torch.Size([5]), integers, H_l, W_l.
        self.up_lr_hw = (self.hr_hw.repeat(self.n_up_factors) / self.up_factors).to(torch.int)

        # Create meta aggregation dictionary once, as global variable
        self.up_lr_to_hr_dict = self.make_aggregation_dictionaries()

        # CHECKPOINT: Test if self.hr_hw can be divided by ALL upscaling factors without a remainder. 
        # Check because this implementation only applies to subset cases.
        if torch.all(self.up_lr_hw.to(torch.float).isclose(self.hr_hw.repeat(self.n_up_factors) / self.up_factors)) == False:
            raise ValueError("Upscaling factors and Height/Width of pixel are not compatible. Use a different algorithm or a subset of compatible upscaling factors.")    

        # Hyperparameters initialisation
        self.gp_hyper_lambda_s = initial_hyperparameters[0].to(device).requires_grad_(False)
        self.gp_hyper_lambda_p = initial_hyperparameters[1].to(device).requires_grad_(False)
        self.gp_hyper_sigma_f = initial_hyperparameters[2].to(device).requires_grad_(False)# amplitude

        # Noise and constant mean, not treated as hyperparameters currently
        self.gp_hyper_noise = torch.tensor([0.05], device = device)
        self.gp_mu = torch.tensor([- 0.8], device = device)

    #############
    ### UTILS ###
    #############

    def scale_individually(self, batch_og):
        """ Scale each scene indiviudally so that each scene has min -10 and max 10 (range 20). Return range used for scaling of each tensor and also minimum of each tensor to be able to project errors and predictions back into the origianal space.

        Args:
            batch (torch.tensor(size = [N, C = 1, H, W])): input tensor

        Returns:
            torch.tensor(size = [N, C = 1, H, W]): output tensor, scaled
            torch.tensor(size = [N, C = 1]): range tensors for each scene
            torch.tensor(size = [N, C = 1]): min tensor for each scene
        """
        batch = torch.clone(batch_og)
        # keep N but flatten other dims
        n_min = torch.min(batch.reshape(batch.shape[0], -1), dim = -1).values
        n_range = torch.max(batch.reshape(batch.shape[0], -1), dim = -1).values - torch.min(batch.reshape(batch.shape[0], -1), dim = -1).values
        # Flatten to [N, -1] and unsqueeze min to [N, 1]
        n_out = torch.div(
            (torch.sub(
             input = torch.flatten(batch, start_dim = 1), other = n_min.unsqueeze(1))), 
             n_range.unsqueeze(1))
        # normalised to 0 to 1 for each aux

        # multiply by 20 to range 0, 20
        n_out = torch.mul(input = n_out, other = 20.0)

        # Subtract 10 to centre at 0
        n_out = torch.sub(input = n_out, other = 10.0)

        # reshape to input shape
        n_out = n_out.reshape(shape = batch.shape)
    
        return n_out, n_range.unsqueeze(1), n_min.unsqueeze(1)
    
    def scale_back_individually(self, batch_mean_scaled, batch_variance_scaled, batch_lml_scaled, batch_scaling_range, batch_scaling_min):
        """_summary_

        Args:
            batch_mean_scaled (_type_): _description_
            batch_variance_scaled (_type_): _description_
            batch_lml_scaled (_type_): _description_
            batch_scaling_range (dict): _description_
            batch_scaling_min (dict): _description_
        """
        # Initialise empty tensors with target shape
        batch_mean = torch.empty(size = batch_mean_scaled.shape)
        batch_variance = torch.empty(size = batch_variance_scaled.shape)

        for u_index, u in enumerate(self.up_factors):
            # extract scaling params
            # tensor shaped [B, C = 1, 60, 60], flatten to [B, -1]
            # batch_scaling_range and batch_scaling_min are tensors shaped [B, 1]

            # 0. Flatten scaled tensors
            # 1. Add 10.0
            # 2. Divide by 20.0
            # 3. Multiply by range
            # 4. Add minimum 
            # 5. reshape
            # 6. Assign to target tensor

            batch_mean[u_index] = torch.add(
                torch.mul(
                    torch.div(
                        torch.add(
                            torch.flatten(
                                batch_mean_scaled[u_index], start_dim = 1), 
                                10.0), 
                                20.0), 
                                batch_scaling_range[u.item()]), 
                                batch_scaling_min[u.item()]).reshape(shape = batch_mean_scaled[u_index].shape)

            batch_variance[u_index] = torch.mul(
                torch.div(
                    torch.add(
                        torch.flatten(
                            batch_variance_scaled[u_index], start_dim = 1), 10.0), 20.0), batch_scaling_range[u.item()]).reshape(shape = batch_variance_scaled[u_index].shape)

        return batch_mean, batch_variance


    def upscale(self, bed_ground_truth, device = device):
        """ Upscale (increase scale of each pixel, reduce resolution) to articially generate low-resolution input. 
        Controlled experiment.

        Args:
            bed_ground_truth (torch.tensor(size = [N, C, H_h, W_h]) with C = 1): ground truth tensor at high resolution
                e.g. torch.Size([300, 1, 60, 60])

        Returns:
            dictionary of len n_up_factors where each entry is a torch.tensor(size = [N, C, H_l, W_l]) with C = 1. 
                The up_factors are the keys
        """
        # Initialize and empty dictionary on device
        up_train_lr = dict(device = device)
        # Iterate through upscaling factors
        for u in self.up_factors:
            # define upscaling function with torch https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html, 
            # Input is (N, C, H_h, W_h) and output is (N, C, H_l, W_l). default settings
            upscaling_function = torch.nn.AvgPool2d(kernel_size = u.item())
            # Add entry to dictionary
            up_train_lr[u.item()] = upscaling_function(bed_ground_truth)
        
        # DICT can't be put on cuda
        return up_train_lr
    
    def set_hypers(self, new_hyperparameters):
        """Update hyperparameters stored in object.

        Args:
            new_hyperparameters (torch.tensor(size = [3, 1])): lambda_s, lambda_p, lambda_f, 
                must be .to(torch.float)): _description_
        """
        self.gp_hyper_lambda_s = new_hyperparameters[0].to(device).requires_grad_(True)
        self.gp_hyper_lambda_p = new_hyperparameters[1].to(device).requires_grad_(True)
        self.gp_hyper_sigma_f = new_hyperparameters[2].to(device).requires_grad_(True)

    def make_aggregation_dictionaries(self):
        """Generates aggregation dictionaries once.

        Returns:
            dict(): Dictionary of dictionaries. 
            meta-level keys are up_factors (u.itemn())
        """

        # Initialise meta_dictionary to hold aggregation dictionary
        # for each up_factor
        up_lr_to_hr_dict = {}

        # u_index is an index int and u is a tensor. u.item() returns the int
        for u_index, u in enumerate(self.up_factors):

            # print(u_index)
            # print(u.item())

            # retrieve lr_hw tensor for the respective u from self
            lr_hw = self.up_lr_hw[u_index].to(device)
            # create aggregation (lr) target indices (e.g. 0 to 899)
            # covariance matrix has squared indices (pairwise)
            lr_flat_indices = torch.arange(
                start =  0, 
                end = lr_hw**2, 
                device = device)
            
            # Indicate row breaks of lr w.r.t. the hr covariance matrix
            # (e.g. 0, 120, 240) going from 30**2 to 60**2
            # includes 0 and last index and intevals are u * hr_hw wide
            lr_row_breaks = torch.linspace(
                start = 0, 
                end = self.hr_hw.item()**2, 
                steps = int(lr_hw + 1), 
                dtype = int, 
                device = device)
            
            # create empty dictionary that maps every lr index 
            # to the corresponding hr indices
            lr_to_hr_dict = dict()

            # start with -1 as it will be updated in first iteration
            lr_row_counter = -1
            
            # interate over all lr covariance indices to create the 
            # dictionary entry
            for i in lr_flat_indices:
                # column counter per row
                j = i % lr_hw

                # check for row breaks
                if j == 0:
                    # Update in very first iteration 
                    lr_row_counter += 1
                    row_base = lr_row_breaks[lr_row_counter]

                # generate list of hr_indices that correspond to each i/j
                corresponding_hr_indices = []


                for k in range(u.item()):
                    # each list is spanning upscaling_factor x rows with 
                    # upscaling_factor x elements each
                    corresponding_hr_indices.extend(
                        range((j * u + (k * self.hr_hw) + row_base),
                            (j * u + (k * self.hr_hw) + u + row_base)))

                lr_to_hr_dict[i.item()] = corresponding_hr_indices

            # Assign dictionary to meta-dictionary for every up_factor   
            up_lr_to_hr_dict[u.item()] = lr_to_hr_dict

        return up_lr_to_hr_dict
    
    #############
    ### BATCH ###
    #############

    def predict_batch(self, batch):
        """Prediction over batches.

        Args:
            batch (torch.tensor(size = [N, 2, H_h, W_h]) where N is the batch size): two channel
                batch_size may change with data_loader iteration

        Returns:
            _type_: _description_
        """
        # Extract bed elevation channel: [N, C, H_h, W_h]. unsqueeze() retains explicit channel dimension
        train_bed_ground_truth = batch[:, 0, :, :].unsqueeze(1).to(device) # channel 0 contains bed elevation

        # Upscale to create dictionary holding low resolution input tensors [N, C, H_l, W_l]
        # DICT can't be put on cuda
        train_bed_lr = self.upscale(
            train_bed_ground_truth, device = device)
        
        # Initialise dictionary
        train_bed_lr_scaled = dict()
        # Scaling ranges are different for different magnification factors
        batch_scaling_range = dict()
        batch_scaling_min = dict()

        ### SCALE LR and maintain scaling params ###
        for u in self.up_factors:
            train_bed_lr_scaled[u.item()], batch_scaling_range[u.item()], batch_scaling_min[u.item()] = self.scale_individually(train_bed_lr[u.item()])

        # Extract auxiliary channel [N, C, H_h, W_h]
        # Currently disgarding scaling parameters.
        # Same res for all.
        train_sur_hr_aux, _ , _ = self.scale_individually(
            batch[:, 1, :, :].unsqueeze(1).to(device)) # channel 1 contains surface elevation

        ### BASE COVARIANCE ###
        # Generate base covariance_matrices using currently stored hypers
        # Input: train_sur_hr_aux (spatial is arbitrary)
        k_ah_ah = self.composite_base_covariance(
            train_sur_hr_aux, device = device).to(device)

        ### AGGREGATE ###
        # Pass in batch of base covariances, returns dict
        k_ah_al, k_al_al = self.aggregate_base_covariance(
            k_ah_ah, device = device)

        ### PREDICTIVE ###
        # TODO: LML
        batch_mean_scaled, batch_variance_scaled, batch_lml_scaled = self.predictive_distribution(
            k_ah_ah, k_ah_al, k_al_al, train_bed_lr_scaled, device)
        
        batch_mean, batch_variance = self.scale_back_individually(batch_mean_scaled, batch_variance_scaled, batch_lml_scaled, batch_scaling_range, batch_scaling_min)
        
        # Reproject: Add 10, divide by 20, multiply by range, add min
        # print(batch_mean_scaled.shape)
        # print(torch.add(batch_mean_scaled, 10.0).shape)
        # print(torch.div(torch.add(batch_mean_scaled, 10.0), 20.0).shape)
        
        # batch_mean = torch.add(
        #     torch.matmul(
        #         torch.flatten(torch.div(torch.add(batch_mean, 10.0), 20.0), start_dim = -2), 
        #         torch.tile(batch_scaling_range.unsqueeze(0), dims = (self.n_up_factors, 1, 1))), 
        #     batch_scaling_min)

        # batch_variance = torch.matmul(torch.div(torch.add(batch_variance, 10.0), 20.0), batch_scaling_range)
        
        ### DELETION ###
        for u in self.up_factors:
                del k_ah_al[u.item()], k_al_al[u.item()]
                torch.cuda.empty_cache()
        
        del k_ah_ah, k_ah_al, k_al_al
        torch.cuda.empty_cache()

        return batch_mean_scaled, batch_variance_scaled, batch_lml_scaled, batch_mean, batch_variance
    
    #################
    ### BASELINE ####
    #################

    def bilinear_interpolation_baseline(self, bed_lr, device):
        """Baseline method

        Args:
            bed_lr (list of torch.tensors): list of torch tensors of different dimensionalities
        """
        # bed_lr is a list of tensors

        # Create one target grid: outer boundries of each scene are the same for both hr and lr: [-1, 1]
        d = torch.linspace(start = (-1.0 + (2 / self.hr_hw)/2).item(),
                           end = (1.0 - (2 / self.hr_hw)/2).item(),
                           steps = self.hr_hw.item(), 
                           device = device)
        
        meshx, meshy = torch.meshgrid((d, d), indexing = "xy") # create mesh
        target_grid = torch.stack((meshx, meshy), 2) # x,y order
        target_grid = target_grid.unsqueeze(0).to(device) # add batch dim: torch.Size([1, self.hr_hw, self.hr_hw, 2])

        # Create empty placeholder tensor of shape [Up = 0, N, C = 1, H, W]

        up_n_bed_hr = torch.empty(
            size = (0, 
                    bed_lr[self.up_factors[0].item()].shape[0], 
                    1, 
                    self.hr_hw.item(), 
                    self.hr_hw.item()),
                    device = device)

        # Simple baseline in bilinear interpolation using torch https://pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html 
        for u_index, u in enumerate(self.up_factors):

            # copy target grid n times
            n = bed_lr[u.item()].shape[0]
            n_target_grids = torch.tile(
                target_grid, dims = (n, 1, 1, 1)).to(device)

            hr_bilinear = torch.nn.functional.grid_sample(
                bed_lr[u.item()].float().to(device), 
                n_target_grids.float(), 
                mode = 'bilinear', 
                # mode = 'nearest', 
                # mode = 'bicubic', 
                padding_mode = 'border', 
                align_corners = False)
            
            up_n_bed_hr = torch.cat((up_n_bed_hr, hr_bilinear.unsqueeze(0).to(device)), dim = 0).to(device) # generate explicit first dim for all up_factors

        return(up_n_bed_hr) # [Up, N, C = mean, H, W]
    
    def bicubic_interpolation_baseline(self, bed_lr, device):
        """Baseline method

        Args:
            bed_lr (list of torch.tensors): list of torch tensors of different dimensionalities
        """
        # bed_lr is a list of tensors

        # Create one target grid: outer boundries of each scene are the same for both hr and lr: [-1, 1]
        d = torch.linspace(start = (-1.0 + (2 / self.hr_hw)/2).item(),
                           end = (1.0 - (2 / self.hr_hw)/2).item(),
                           steps = self.hr_hw.item(), 
                           device = device)
        
        meshx, meshy = torch.meshgrid((d, d), indexing = "xy") # create mesh
        target_grid = torch.stack((meshx, meshy), 2) # x,y order
        target_grid = target_grid.unsqueeze(0).to(device) # add batch dim: torch.Size([1, self.hr_hw, self.hr_hw, 2])

        # Create empty placeholder tensor of shape [Up = 0, N, C = 1, H, W]

        up_n_bed_hr = torch.empty(
            size = (0, 
                    bed_lr[self.up_factors[0].item()].shape[0], 
                    1, 
                    self.hr_hw.item(), 
                    self.hr_hw.item()),
                    device = device)

        # Simple baseline in bilinear interpolation using torch https://pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html 
        for u_index, u in enumerate(self.up_factors):

            # copy target grid n times
            n = bed_lr[u.item()].shape[0]
            n_target_grids = torch.tile(
                target_grid, dims = (n, 1, 1, 1)).to(device)

            hr_bilinear = torch.nn.functional.grid_sample(
                bed_lr[u.item()].float().to(device), 
                n_target_grids.float(), 
                # mode = 'bilinear', 
                mode = 'bicubic', 
                padding_mode = 'border', 
                align_corners = False)
            
            up_n_bed_hr = torch.cat((up_n_bed_hr, hr_bilinear.unsqueeze(0).to(device)), dim = 0).to(device) # generate explicit first dim for all up_factors

        return(up_n_bed_hr) # [Up, N, C = mean, H, W]
    
    ###############
    ### METRICS ###
    ###############

    def rmse(self, ground_truth, predictions):
        """_summary_

        Args:
            ground_truth (_type_): [N, 1, H, W] - will be copied for all up_factors
            predictions (_type_): [Up, N, 1, H, W]

        Returns:
            [Up, N, 1]
        """
        n = predictions.shape[0]
        # Copy for n_Up_factors into [Up, N, 1, H, W]
        n_ground_truth = torch.tile(ground_truth.unsqueeze(0), dims = (n, 1, 1, 1, 1))

        error = torch.sub(n_ground_truth, predictions) # subtract elementwise
        squared_error = torch.pow(error, exponent = 2) # square error to eliminate negatives
        mean_squared_error = torch.mean(squared_error, dim = (-2, -1)) # mean across H and W (last two dim)
        root_mean_squared_error = torch.sqrt(mean_squared_error)
        
        # ToDo: mean over N
        return root_mean_squared_error # [Up, N, C = RMSE]
    
    def psnr(self, ground_truth, predictions, max_signal):
        """_summary_

        Args:
            ground_truth (_type_): [N, 1, H, W] - will be copied for all up_factors
            predictions (_type_): [Up, N, 1, H, W] - in original space
            max_signal (torch.tensor(size = [1]))

        Returns:
            [Up, N, 1]
        """
        # Call RSME which returns shape [Up, N, C = RMSE]
        rmse = self.rmse(ground_truth, predictions)
        # square rmse to get mse
        mse = torch.pow(rmse, exponent = 2)
        # check-point
        print(mse.shape)

        # base 10 logarithm
        psnr = torch.mul(
            input = torch.log10(torch.div(
                torch.pow(max_signal, exponent = 2), 
                mse)), 
            other = 10.0)
        
        return psnr # [Up, N, C = PNSR]
    
    def nll(self, batch_ground_truth, batch_mean, batch_variance, device):
        """Return one mean nll value for each upfactor and each scene in batch.
        smaller is better: larger likelihood is better, thus smaller neg. likelihood is better

        Args:
            batch_ground_truth (toch.tensor(size = [N, C, H, W])): true values should be the same for all up_factors
            batch_mean (toch.tensor(size = [Up, N, C, H, W])): mean predictions
            batch_variance (toch.tensor(size = [Up, N, C, H, W])): predicted variances. Should be > 0 for all.

        Returns
            (toch.tensor(size = [Up, N, C])): with C = 1 and n = Nb (batches)
        """
        up_nll = torch.empty(size = (0, batch_ground_truth.shape[0], 1), device = device)
        
        for u_index, u in enumerate(self.up_factors):
            
            # Target are ground truth samples, input are Gaussian mean expectation, and var are the variances
            # reduction = "mean" returns one value for each upfactor: 
            # use reduction = "none" and average dims as requried to keep N images
            nll = torch.nn.functional.gaussian_nll_loss(input = batch_mean[u_index], 
                                                        target = batch_ground_truth, # stays the same
                                                        var = batch_variance[u_index], full = False, eps = 1e-06, reduction = 'none').to(device)
            
            # Average across h & W dimensions for every image and unsqueeze for explicit first dim
            up_nll = torch.cat((up_nll, torch.mean(nll, dim = (2, 3)).unsqueeze(0)), dim = 0)

        return up_nll
        
    ##################
    ### COVARIANCE ###
    ##################

    def composite_base_covariance(self, batch_aux_tensor, device):
        """Takes in a batch of aux_tensors and produces the pairwise covaraince matricex for the batch.
        The spatial base covariance part of the product compositive kernel is the same for all N thus we are creating N copies for all batch members. 
        The pixel base covariance is calculated over batches. 

        Args:
            batch_aux_tensor (torch.tensor(size = [N, C, H_h, W_h])): with N = batch_size and C is one channel

        Returns:
            torch.tensor(size = [N, C, H_h **2, W_h ** 2]): base covariance matrix k_ah_ah
        """
        # spatial_base_covariance does not depend on any inputs

        # Pass batch size into spatial covariance function
        # Adds 0.5 GB
        spatial_base_covariance_matrix = self.spatial_base_covariance(batch_aux_tensor.shape[0], device).to(dtype = torch.float32)

        # Adds 1.5 GB
        pixel_base_covariance_matrix = self.pixel_base_covariance(batch_aux_tensor, device)
            
        # Element-wise multiplication, Hadamard product, AND operation
        product_base_covar = torch.mul(spatial_base_covariance_matrix, pixel_base_covariance_matrix)

        # In-place and combined
        # product_base_covar = (spatial_base_covariance_matrix.mul_(pixel_base_covariance_matrix)).mul_(self.gp_hyper_sigma_f)
            
        # multiply all elements in matrix with same scalar, in-place
        product_base_covar = torch.mul(product_base_covar, self.gp_hyper_sigma_f)

        # product_base_covar = product_base_covar.mul_(self.gp_hyper_sigma_f)

        del spatial_base_covariance_matrix, pixel_base_covariance_matrix, batch_aux_tensor
        torch.cuda.empty_cache()

        return product_base_covar
    
    def only_spatial_base_covariance(self):
        # Fast do no batching needed
        spatial_base_covariance_matrix = self.spatial_base_covariance(self.n_train)

        return spatial_base_covariance_matrix

    def spatial_base_covariance(self, n_batch, device):
        """ Calculate spatial covariance using n_batches, self.hr_hw and self.gp_hyper_lambda_s for the scaling.
        Smooth, sparse and local.
        Lambda_s is the hp that controls the receptive field.
        Same for all n so we calculate it once and create copies.

        Args:
            n_batch (int): number of copies we need
            device (object): cuda or cpu

        Returns:
            torch.tensor(size = [N, C, H_h ** 2, W_H ** 2]): with N = n_batches and C = 1 which is the covariance channel.
        """
        # Normalised mid_points of all pixels in scene
        xs = torch.arange(0, (self.hr_hw.item())).repeat(self.hr_hw.item(), 1).to(device)
        ys = xs.mT
        # x and y dim of midpoints from 0 to 1 
        mid_points_norm = torch.cat((ys.unsqueeze(0), xs.unsqueeze(0)), dim = 0)/(self.hr_hw - 1) # torch.Size([2, 60, 60])

        # Flatten the last two dims
        mid_points_flat = torch.flatten(mid_points_norm, start_dim = -2) # torch.Size([2, 3600])
        # broadcast to calculate pairwise distance
        dist = torch.sub(mid_points_flat.unsqueeze(-1), mid_points_flat.unsqueeze(-2))
        # square all distances
        dist_square = torch.pow(dist, exponent = 2)
        # sum x direction dist and y direction dist (Euclidean dist)
        dist_sum = torch.sum(dist_square, dim = 0)
        dist_euc = torch.sqrt(dist_sum) # torch.Size([3600, 3600]), Pythagoras, max is 1.4142 (corners, sqrt(2))

        Z = torch.div(dist_euc, self.gp_hyper_lambda_s) # divide by scalar, lambda_s is threshold for 0 covariance
        # Mask large Z's (too-far-away values) with nan before replacing values with zero later
        Z[Z >= 1] = float('nan')
        # first term pushes small distances to 0 and distances near 1 close to zero
        # second terms is clipped at 1 so that small distances will approach 1
        prod_term1 = torch.pow(torch.add(-Z, 1.), exponent = 3) # (1 - Z) = (-Z + 1)
        prod_term2 = torch.add(torch.mul(Z, 3.), 1.)
        cov_matrix = torch.mul(prod_term1, prod_term2) # elementwise multiplication

        # fill nan's with zero
        cov_matrix[torch.isnan(cov_matrix)] = 0.0
        cov_matrix = cov_matrix.unsqueeze(0).unsqueeze(1) # torch.Size([1, 1, 3600, 3600])

        # Create n copies
        n_cov_matrix = torch.tile(cov_matrix, dims = (n_batch, 1, 1, 1)) # torch.Size([n, 1, 3600, 3600])

        # Clean-up
        del xs, ys, mid_points_norm, mid_points_flat, dist, dist_square, dist_sum, dist_euc, Z, prod_term1, prod_term2, cov_matrix, n_batch
        torch.cuda.empty_cache()

        # torch.Size([8, 1, 3600, 3600])
        return n_cov_matrix

    def pixel_base_covariance(self, batch_aux_tensor, device):
        """coupling(decoupling) of similar(dissimilar) pixel values, non-stationary
        lambda_p is the lengthscale of the RBF kernel, default is 0.6931 in GPytorch
        for 300 scenes this has a wall time of 8:20 min on MacPro
        Currently handles full batch at a time.
        Overwriting of variable names to reduce memory.

        Args:
            batch_aux_tensor (torch.tensor(size = [N, C, H_h, W_h])): a batch of the auxiliary variable
            device (object): cuda or cpu

        Returns:
            torch.tensor(size = [N, C, H_h **2, W_h **2]): pixel covaraince
        """
        # Flatten x & y / reduce the last two dimensions and overwrite
        batch_aux_tensor = torch.flatten(batch_aux_tensor, start_dim = -2).to(device)

        ### Do element-wise if it crashes ###

        # Overwriting for memory
        # DISTANCE: broadcast for pairwise dist (in pixel dim): torch.Size([5, 1, 3600, 1]), torch.Size([5, 1, 1, 3600])
        dist = torch.sub(batch_aux_tensor.unsqueeze(-1), batch_aux_tensor.unsqueeze(-2))

        # DISTANCE SQUARED: direction of distance does not matter, overwrite for memory
        dist = torch.pow(dist, exponent = 2)
        # DISTANCE SQUARED AND SCALED: 
        dist = torch.div(dist, (2 * torch.pow(self.gp_hyper_lambda_p, exponent = 2)))

        # Exponent will evaluate to 1 if there is zero distance
        rbf = torch.exp( - dist)

        # Clean-up
        del dist, batch_aux_tensor
        torch.cuda.empty_cache()

        return rbf
    
    def aggregate_base_covariance(self, batch_base_covariance_matrix, device):
        """Aggregate k_ah_ah to k_ah_al (column aggregation) and then k_al_al
        using the global aggregation dictionary created once at 
        initialisation of object.

        Args:
            batch_base_covariance_matrix (torch.tensor(size = [N, C, H_h, W_h])): 
                k_ah_ah
            device (object): cuda or cpu

        Returns:
            dict: Dictionary with a torch.tensor(size = [N, 1, H_h, W_l]) 
                  for each up_factor. Tensors on device.
            dict: Dictionary with a torch.tensor(size = [N, 1, H_l, W_l]) 
                  for each up_factor. Tensors on device.
        """

        # Intialise empty dictionaries which will have one tensor entry per up_factor
        up_k_ah_al = dict()
        up_k_al_al = dict()

        for u_index, u in enumerate(self.up_factors):
            # use nested list of all dictionary entries for respective up_factor
            # to subset columns of covariance matrix.
            # Simple subset: average over last dimension (equally weighted). 
            # cardinality of last dim changes: for up = 2: average over 4 values
            up_k_ah_al[u.item()] = torch.mean(
                batch_base_covariance_matrix[:, :, :, list(self.up_lr_to_hr_dict[u.item()].values())], 
                dim = -1).to(device)
            
            # apply .mt to transpose the last two dims and repeat
            up_k_al_al[u.item()] = torch.mean(
                up_k_ah_al[u.item()].mT[:, :, :, list(self.up_lr_to_hr_dict[u.item()].values())], 
                dim = -1).to(device)

        del batch_base_covariance_matrix
        torch.cuda.empty_cache()
            
        return up_k_ah_al, up_k_al_al
    
    ##################
    ### PREDICTIVE ###
    ##################
    
    def predictive_distribution(self, k_ah_ah, k_ah_al, k_al_al, train_lr, device):
        """Calculation of the mean and variance of the Gaussian predictive distribution as well as of Log Marginal Likelihood. 
        Here we implement the inversion through the Cholesky decomposition [torch.linalg.cholesky()] followed by the inversion of the Cholesky [torch.cholesky_inverse(L)]. 
        This is slighly different to the Algorithm 2.10 proposed in Rasmussen & Williams but more suitable to implement the weight correction performed by Reid et al.
        - Loop over up_factors, but operate on batch at once.

        k_ah_ah is a torch tensor.
        k_ah_al is a list of tensors.
        k_al_al is a list of tensors.
        train_lr is the lr for the mean
        """
        # Go through one at a time
        # n_base_covars = k_ah_ah.shape[0]

        # empty tensor
        up_n_mean = torch.empty(size = (0, k_ah_ah.shape[0], 1, self.hr_hw, self.hr_hw), device = device)
        up_n_variance = torch.empty(size = (0, k_ah_ah.shape[0], 1, self.hr_hw, self.hr_hw), device = device)
        up_n_lml = torch.empty(size = (0, k_ah_ah.shape[0]), device = device)

        for u_index, u in enumerate(self.up_factors):

            # both L and k_inv are used throughout
            L = torch.linalg.cholesky(
                torch.add(k_al_al[u.item()],
                          torch.tile(
                              (torch.eye(
                                  n = k_al_al[u.item()].shape[-1], device = device) 
                                  * self.gp_hyper_noise).unsqueeze(0).unsqueeze(0), 
                                  dims = (k_al_al[u.item()].shape[0], 1, 1, 1))))

            k_inv = torch.cholesky_inverse(L).to(device)

            #### Mean ###

            pl_al_minus_mu = torch.sub(train_lr[u.item()], 
                                       torch.mul(torch.ones(size = train_lr[u.item()].shape, device = device), 
                                                 self.gp_mu)).to(device)
            W = torch.matmul(k_ah_al[u.item()], k_inv).to(device)

            ### Correction ###
            # torch.div is element-wise vision
            # Numerator: W * pl_minus: torch.Size([6, 1, 3600, 900]) * torch.Size([6, 1, 900, 1]) = torch.Size([6, 1, 3600, 1])
            # Denominator: W * ones: torch.Size([6, 1, 3600, 900]) * torch.Size([900, 1]) = torch.Size([6, 1, 3600, 1])
            # w/o correcation: no division, just numerator
            # n_mean = torch.div(
            #     torch.matmul(W, torch.flatten(pl_al_minus_mu, start_dim = -2).unsqueeze(-1)), 
            #     torch.matmul(W, torch.ones(size = (W.shape[-1], 1), device = device))) + self.gp_mu
            
            # W/O correction overwrite
            n_mean = torch.matmul(W, torch.flatten(pl_al_minus_mu, start_dim = -2).unsqueeze(-1)) + self.gp_mu
            
            # cast into 2D shape: N, C, H, W
            n_mean = n_mean.reshape(n_mean.shape[0], 1, int(np.sqrt(n_mean.shape[- 2])), -1)

            ### Covariance ###
            # .mT transposes the last two dims of a matrix
            n_covariance = k_ah_ah - torch.matmul(k_ah_al[u.item()], 
                                                  torch.matmul(k_inv, k_ah_al[u.item()].mT))

            # covariance is too memory intensive. Return only variance.
            # extract variances (on diagonal) from covariance matrix and reshape to square
            n_variance = torch.diagonal(n_covariance, dim1 = -2, dim2 = -1).reshape(
                n_covariance.shape[0], 
                n_covariance.shape[1],
                int(math.sqrt(n_covariance.shape[3])),
                -1)

            ### LML ###
            # 2.30 in Rasmussen
            # https://d2l.ai/chapter_gaussian-processes/gp-inference.html
    
            # Term1: Kernel term
            term1 = torch.mul(torch.matmul(torch.flatten(train_lr[u.item()].mT, start_dim = -2).unsqueeze(-2), torch.matmul(k_inv, torch.flatten(train_lr[u.item()].mT, start_dim = -2).unsqueeze(-1))), 0.5)

            # Term2: Determinant term. 2 and 0.5 cancel each other out, sum in log space, log makes values negative
            term2 = torch.sum(torch.log(torch.diagonal(L, dim1 = -2, dim2 = -1)), dim = (1, 2)).to(device)

            # Need trick https://math.stackexchange.com/questions/3158303/using-cholesky-decomposition-to-compute-covariance-matrix-determinant
            # Does not work: term2 = torch.mul(torch.log(torch.linalg.det(k_al_al + (torch.eye(n = k_al_al.shape[-1]) * noise))), 0.5).reshape(1, 1)
            # Flatten shape and extract n, natural log
    
            # Term3: Constant term: Only extracts shape from train_lr
            term3 = (torch.log(torch.tensor(2 * math.pi, device = device)) * torch.flatten(train_lr[u.item()], start_dim = - 2).shape[-1] * 0.5).reshape(-1)
            n_lml = (- term1.reshape(-1) - term2 - term3).to(device)

            up_n_mean = torch.cat((up_n_mean, n_mean.unsqueeze(0)), dim = 0)
            # Returning full covariance is too expensive
            up_n_variance = torch.cat((up_n_variance, n_variance.unsqueeze(0)), dim = 0)
            up_n_lml = torch.cat((up_n_lml, n_lml.unsqueeze(0)), dim = 0)

        return up_n_mean, up_n_variance, up_n_lml

## Scaling

In [10]:
hr_hw = torch.tensor([60])
up_factors = torch.tensor([2, 3, 4, 5, 6])
initial_hyperparameters = torch.tensor([[1.4], [0.3], [0.7]]).to(torch.float)
initial_hyperparameters = torch.tensor([[1.0], [0.3], [1.0]]).to(torch.float)

# Initialise srGP object
srGP = Superresolution_GP(hr_hw, up_factors, initial_hyperparameters, device)

# Initialise scaled tensor as copy of training tensor
scaled_training_tensor = torch.clone(training_tensor)

# Scale bed elevation 
scaled_training_tensor[:, 0, :, :], scaling_n_range, scaling_n_min = srGP.scale_individually(
    training_tensor[:, 0, :, :])
scaled_training_tensor[:, 1, :, :], aux_scaling_n_range, aux_scaling_n_min = srGP.scale_individually(
    training_tensor[:, 1, :, :])

print("Mean after scaling: ", torch.mean(scaled_training_tensor[:, 0, :, :]).item())
print("Aux mean after scaling: ", torch.mean(scaled_training_tensor[:, 1, :, :]).item())

Mean after scaling:  -0.08383277803659439
Aux mean after scaling:  0.14109228551387787


In [11]:
# Correlations
torch.corrcoef(torch.cat(
    (scaled_training_tensor[:, 0, :, :].reshape(1, -1),
    scaled_training_tensor[:, 1, :, :].reshape(1, -1)),
    dim = 0))

# The correlation between scaled bed elevation and surface elevation is low generally

torch.corrcoef(torch.cat(
    (scaled_training_tensor[:, 0, :, :].reshape(1, -1),
    training_tensor[:, 1, :, :].reshape(1, -1)),
    dim = 0))

tensor([[ 1.0000, -0.0166],
        [-0.0166,  1.0000]], device='cuda:0')

In [12]:
torch.mean(scaling_n_range)

fig = go.Figure(data = [go.Histogram(x = scaling_n_range.squeeze().detach().cpu().numpy())])
fig.update_layout(title = "Distribution of scaling ranges across scenes")
fig.show()

# One image has a huge range 

In [13]:
# Reconstruction example
# Add 10 -> [0.0, 20.0]
# Divide by 20 -> [0.0, 1.0]
# Multiply by range
# torch.add(torch.mul(torch.flatten(torch.div(torch.add(scaled_training_tensor[:, 0, :, :], 10.0), 20.0), start_dim = -2), scaling_n_range), scaling_n_min).reshape(300, 60, 60)

# Predictions

In [29]:
### Hyperparameters ###
hr_hw = torch.tensor([60])
up_factors = torch.tensor([2, 3, 4, 5, 6])
# initial_hyperparameters = torch.tensor([[0.4], [0.9], [1.0]]).to(torch.float)
# Without correction: initial_hyperparameters = torch.tensor([[0.56], [7.0], [16.63]]).to(torch.float)
initial_hyperparameters = torch.tensor([[0.9], [3.2], [19.0]]).to(torch.float)

# Initialise srGP object
srGP = Superresolution_GP(hr_hw, up_factors, initial_hyperparameters, device)

### Baseline in original domain ###
n_bilinear_rmse = srGP.rmse(
        training_tensor[:, 0, :, :].unsqueeze(1).to(device), 
        srGP.bilinear_interpolation_baseline(
            srGP.upscale(
                training_tensor[:, 0, :, :].unsqueeze(1).to(device), device = device),
                device)) # [Up, N, C = RMSE]

bilinear_rmse = torch.mean(n_bilinear_rmse, dim = (1, 2))
print("Bilinear RMSE [meters]: {}".format(bilinear_rmse))

n_bicubic_rmse = srGP.rmse(
        training_tensor[:, 0, :, :].unsqueeze(1).to(device), 
        srGP.bicubic_interpolation_baseline(
            srGP.upscale(
                training_tensor[:, 0, :, :].unsqueeze(1).to(device), device = device),
                device)) # [Up, N, C = RMSE]

bicubic_rmse = torch.mean(n_bicubic_rmse, dim = (1, 2))
print("Bicubic RMSE [meters]: {}".format(bicubic_rmse))

# Pass in unscaled data. It will scale inside function
dataloader = torch.utils.data.DataLoader(training_tensor[:, :, :, :], 
                                         batch_size = 8, 
                                         shuffle = False)

# [Up, N = 0, C]
n_rmse = torch.empty(size = (srGP.n_up_factors, 0, 1), 
                     device = device)
n_nll = torch.empty(size = (srGP.n_up_factors, 0, 1), 
                     device = device)

for batch in dataloader:
    up_nb_mean_scaled, up_nb_variance_scaled, up_nb_lml_scaled, up_nb_mean, up_nb_variance = srGP.predict_batch(batch)

    # RMSE per batch
    n_rmse = torch.cat(
        (n_rmse, srGP.rmse(
            batch[:, 0, :, :].unsqueeze(1).to(device), 
            up_nb_mean.to(device))),
        dim = 1)
    
    n_nll = torch.cat(
        (n_nll, srGP.nll(
            batch[:, 0, :, :].unsqueeze(1).to(device), 
            up_nb_mean.to(device),
            up_nb_variance.to(device), 
            device)),
        dim = 1)

# average over batches
proposed_rmse = torch.mean(n_rmse, dim = (1, 2))
proposed_nll = torch.mean(n_nll, dim = (1, 2))

print("Proposed RMSE [meters]: {}".format(proposed_rmse))

Bilinear RMSE [meters]: tensor([ 2.0237,  3.8991,  6.2840,  8.5836, 11.1607], device='cuda:0')
Bicubic RMSE [meters]: tensor([1.3613, 2.5354, 4.2613, 6.0752, 8.1305], device='cuda:0')
Proposed RMSE [meters]: tensor([2.1782, 3.2450, 4.4451, 5.9326, 7.7525], device='cuda:0')


In [15]:
# Extract n_rmse for highest upscaling factor
n_rmse[4, :, 0]

tensor([ 1.3123,  0.7331,  0.4868,  1.2078,  0.9504,  0.9667,  2.1145,  2.5053,
         0.8285,  1.9077,  3.8123,  2.7152,  2.2970,  2.3258,  2.0812,  1.9657,
         2.1975,  2.6550,  2.0114,  3.6014,  4.2687,  0.6392,  0.6325,  0.7530,
         0.5802,  2.0818,  2.1237,  1.6018,  0.3641,  1.0313,  1.9419,  2.5695,
         4.5835,  4.3401,  5.3627,  2.2256,  2.6072,  1.8734,  1.5729,  1.4866,
         2.5033,  0.6992,  0.5126,  1.1736,  1.0574,  1.2772,  1.6116,  0.8652,
         0.3928,  1.7244,  3.9824,  3.6187,  2.9848,  4.8140,  7.4399,  3.6411,
         6.3375,  4.7944,  2.9392,  2.3924,  2.1939,  0.7908,  1.1994,  2.3871,
         1.8655,  1.9371,  1.0037,  0.6307,  0.7221,  1.0195,  1.4148,  2.1275,
         0.9595,  4.3609, 21.3370,  9.3977, 10.6576,  4.8276,  4.3558,  4.3235,
         2.6952,  2.3221,  4.1685,  5.5931,  1.9254,  4.0998,  1.1332,  2.2493,
         2.2616,  2.3006,  5.4790,  1.4687,  2.3096,  5.0448,  6.5379,  9.1572,
        10.2752,  5.5657,  5.7342,  7.44

In [16]:
fig = px.imshow(torch.mean(training_tensor_full[:, 2, :, :], dim = (1, 2)).reshape(20, 20).cpu(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "Ice cover areas")
fig.show()

In [17]:
torch.argmin(n_rmse[4, :, 0])
n_rmse[4, :, 0][48]

tensor(0.3928, device='cuda:0')

In [18]:
fig = px.imshow(n_rmse[4, :, 0].reshape(20, 20).cpu(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "RMSE distribution")
fig.show()

In [19]:
fig = px.imshow(n_bicubic_rmse[4, :, 0].reshape(20, 20).cpu(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "RMSE distribution of proposed")
fig.show()

In [20]:
fig = px.imshow(n_bicubic_rmse[4, :, 0].reshape(20, 20).cpu() - n_rmse[4, :, 0].reshape(20, 20).cpu(), 
                color_continuous_scale = 'RdBu_r',
                # DEFAULT for matrices: only tensor because of conversion
                origin = "upper", title = "RMSE distribution")
fig.show()

In [21]:
n_rmse[4, :, 0][45]
n_rmse[4, :, 0][17*20 + 12]
17*20 + 12
3*20 + 8

68

In [22]:
scaling_n_range

tensor([[ 160.2678],
        [  97.9009],
        [  75.9993],
        [ 211.1096],
        [ 138.8430],
        [ 167.6365],
        [ 394.4438],
        [ 395.4771],
        [ 138.2180],
        [ 206.1116],
        [ 269.1589],
        [ 242.8667],
        [ 280.2485],
        [ 391.6069],
        [ 176.0645],
        [ 229.2659],
        [ 343.4749],
        [ 212.4766],
        [ 160.0901],
        [ 349.0212],
        [ 294.3667],
        [ 142.5454],
        [ 121.7300],
        [ 170.2593],
        [  59.4224],
        [ 203.9004],
        [ 405.4524],
        [ 157.7642],
        [  73.8604],
        [ 109.0786],
        [ 209.5759],
        [ 209.0586],
        [ 393.8792],
        [ 411.1096],
        [ 441.0205],
        [ 307.3789],
        [ 264.7051],
        [ 164.3643],
        [ 255.6802],
        [ 224.3787],
        [ 419.4229],
        [ 131.1509],
        [ 103.2886],
        [  80.8445],
        [ 228.9858],
        [ 325.6167],
        [ 358.6174],
        [ 128

In [23]:
def visualise_results(bilinear_rmse_tensor, bicubic_rmse_tensor, proposed_rmse_tensor, n_scenes, domain_name):

    # RMSE
    fig = go.Figure()
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), 
                             y = bilinear_rmse_tensor, 
                             mode = 'lines+markers', 
                             name = "Bilinear baseline"))
    
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), 
                             y = bicubic_rmse_tensor, 
                             mode = 'lines+markers', 
                             name = "Bicubic baseline"))
    
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), 
                             y = proposed_rmse_tensor, 
                             mode = 'lines+markers', 
                             name = "Proposed algorithm"))

    fig.update_layout(
        title = 'Reconstruction loss [RMSE] of proposed vs. baselines - {} scenes near domain {}'.format(n_scenes, domain_name),
        template = "plotly_white")
    
    fig.update_xaxes(
        title_text = 'Upscaling factor')
    
    fig.update_yaxes(
        title_text = 'RMSE')
    
    fig.show()

In [24]:
visualise_results(bilinear_rmse.detach().cpu().numpy(), 
                  bicubic_rmse.detach().cpu().numpy(), 
                  proposed_rmse.detach().cpu().numpy(), 
                  n_scenes = 300, 
                  domain_name = "Transantarctic mountains")

## Optimize hyperparameters

In [28]:
step_size = 0.001
n_epochs = 20

hr_hw = torch.tensor([60])
up_factors = torch.tensor([2, 3, 4, 5, 6])
initial_hyperparameters = torch.tensor([[0.9], [3.54], [19.0]]).to(torch.float)

# Initialise srGP object
srGP = Superresolution_GP(hr_hw, up_factors, initial_hyperparameters, device)
# Turn on requires_grad
srGP.set_hypers(initial_hyperparameters)

for e in range(n_epochs):
    print("Epoch", e)

    # Initialise a dataloader on training_tensor
    dataloader = torch.utils.data.DataLoader(training_tensor[0:300, :, :, :], batch_size = 16, shuffle = False)

    # Send batches through epoch
    for batch in dataloader:
        # Only care about lml 
        up_nb_mean_scaled, up_nb_variance_scaled, up_nb_lml_scaled, up_nb_mean, up_nb_variance = srGP.predict_batch(batch)
        # Loss needs to be a single value so average over all. Loss so will be minimized
        # print(up_nb_lml.shape)
        loss = - torch.mean(up_nb_lml_scaled)
        loss.backward()

        # Update parameters
        srGP.gp_hyper_lambda_s.data = srGP.gp_hyper_lambda_s.data - (step_size * srGP.gp_hyper_lambda_s.grad.data)
        srGP.gp_hyper_lambda_p.data = srGP.gp_hyper_lambda_p.data - (step_size * srGP.gp_hyper_lambda_p.grad.data)
        srGP.gp_hyper_sigma_f.data = srGP.gp_hyper_sigma_f.data - (step_size * srGP.gp_hyper_sigma_f.grad.data)

        # Re setting gradients to zero
        srGP.gp_hyper_lambda_s.grad.data.zero_()
        srGP.gp_hyper_lambda_p.grad.data.zero_()
        srGP.gp_hyper_sigma_f.grad.data.zero_()

        print("NLML:", np.round(loss.detach().cpu().item(), decimals = 2), 
            "lambda_s:", np.round(srGP.gp_hyper_lambda_s.data.detach().cpu().item(), decimals = 2),
            "lambda_p:", np.round(srGP.gp_hyper_lambda_p.data.detach().cpu().item(), decimals = 2),
            "sigma_f:", np.round(srGP.gp_hyper_sigma_f.data.detach().cpu().item(), decimals = 2))

Epoch 0
NLML: 52.59 lambda_s: 1.01 lambda_p: 3.55 sigma_f: 19.0
NLML: 71.97 lambda_s: 1.01 lambda_p: 3.56 sigma_f: 19.0
NLML: 64.19 lambda_s: 1.02 lambda_p: 3.57 sigma_f: 19.0
NLML: 108.48 lambda_s: 0.93 lambda_p: 3.58 sigma_f: 19.0
NLML: 72.43 lambda_s: 0.97 lambda_p: 3.58 sigma_f: 19.0
NLML: 100.6 lambda_s: 0.94 lambda_p: 3.59 sigma_f: 19.0
NLML: 122.93 lambda_s: 0.91 lambda_p: 3.6 sigma_f: 19.0
NLML: 197.16 lambda_s: 0.78 lambda_p: 3.61 sigma_f: 19.0
NLML: 176.88 lambda_s: 0.7 lambda_p: 3.61 sigma_f: 19.0
NLML: 373.78 lambda_s: 0.17 lambda_p: 3.61 sigma_f: 19.01
NLML: 544.24 lambda_s: 2.69 lambda_p: 3.61 sigma_f: 19.0


KeyboardInterrupt: 

Observations:
- lambda_p keeps going up for a bit.
- relatitive to scaling
- should rather compute lml in original domain

With pixel correction:
- lamba_s jumps around a lot (large gradients)
- 

# Optimise for NLL

In [ ]:
# Need larger step size
step_size = 0.001
n_epochs = 20

hr_hw = torch.tensor([60])
up_factors = torch.tensor([2, 3, 4, 5, 6])
initial_hyperparameters = torch.tensor([[1.7], [0.47], [0.3]]).to(torch.float)

# Initialise srGP object
srGP = Superresolution_GP(hr_hw, up_factors, initial_hyperparameters, device)
# Turn on requires_grad
srGP.set_hypers(initial_hyperparameters)

for e in range(n_epochs):
    print("Epoch", e)

    # Initialise a dataloader
    dataloader = torch.utils.data.DataLoader(scaled_training_tensor[0:300, :, :, :], batch_size = 16, shuffle = False)

    # Send batches through epoch
    for batch in dataloader:
        up_nb_mean, up_nb_variance, up_nb_lml = srGP.predict_batch(batch)
        # Loss needs to be a single value so average over all. Loss so will be minimized
        loss = torch.mean(srGP.nll(batch[:, 0, :, :].unsqueeze(1).to(device), 
                         up_nb_mean.to(device), 
                         up_nb_variance.to(device), device))
        loss.backward()

        # Update parameters
        srGP.gp_hyper_lambda_s.data = srGP.gp_hyper_lambda_s.data - (step_size * srGP.gp_hyper_lambda_s.grad.data)
        srGP.gp_hyper_lambda_p.data = srGP.gp_hyper_lambda_p.data - (step_size * srGP.gp_hyper_lambda_p.grad.data)
        srGP.gp_hyper_sigma_f.data = srGP.gp_hyper_sigma_f.data - (step_size * srGP.gp_hyper_sigma_f.grad.data)

        # Re setting gradients to zero
        srGP.gp_hyper_lambda_s.grad.data.zero_()
        srGP.gp_hyper_lambda_p.grad.data.zero_()
        srGP.gp_hyper_sigma_f.grad.data.zero_()

        print("NLL:", np.round(loss.detach().cpu().item(), decimals = 2), 
            "lambda_s:", np.round(srGP.gp_hyper_lambda_s.data.detach().cpu().item(), decimals = 2),
            "lambda_p:", np.round(srGP.gp_hyper_lambda_p.data.detach().cpu().item(), decimals = 2),
            "sigma_f:", np.round(srGP.gp_hyper_sigma_f.data.detach().cpu().item(), decimals = 2))

In [ ]:
# Need larger step size
step_size = 0.2
n_epochs = 40

hr_hw = torch.tensor([60])
up_factors = torch.tensor([2, 3, 4, 5, 6])
initial_hyperparameters = torch.tensor([[1.64], [0.43], [0.43]]).to(torch.float)

# Initialise srGP object
srGP = Superresolution_GP(hr_hw, up_factors, initial_hyperparameters, device)
# Turn on requires_grad
srGP.set_hypers(initial_hyperparameters)

for e in range(n_epochs):
    print("Epoch", e)

    # Initialise a dataloader
    dataloader = torch.utils.data.DataLoader(scaled_training_tensor[0:300, :, :, :], batch_size = 16, shuffle = False)

    # Send batches through epoch
    for batch in dataloader:
        up_nb_mean, up_nb_variance, up_nb_lml = srGP.predict_batch(batch)
        # Loss needs to be a single value so average over all. Loss so will be minimized
        loss = torch.mean(srGP.rmse(batch[:, 0, :, :].unsqueeze(1).to(device), 
                         up_nb_mean.to(device)))
        loss.backward()

        # Update parameters
        srGP.gp_hyper_lambda_s.data = srGP.gp_hyper_lambda_s.data - (step_size * srGP.gp_hyper_lambda_s.grad.data)
        srGP.gp_hyper_lambda_p.data = srGP.gp_hyper_lambda_p.data - (step_size * srGP.gp_hyper_lambda_p.grad.data)
        srGP.gp_hyper_sigma_f.data = srGP.gp_hyper_sigma_f.data - (step_size * srGP.gp_hyper_sigma_f.grad.data)

        # Re setting gradients to zero
        srGP.gp_hyper_lambda_s.grad.data.zero_()
        srGP.gp_hyper_lambda_p.grad.data.zero_()
        srGP.gp_hyper_sigma_f.grad.data.zero_()

        print("RMSE:", np.round(loss.detach().cpu().item(), decimals = 5), 
            "lambda_s:", np.round(srGP.gp_hyper_lambda_s.data.detach().cpu().item(), decimals = 2),
            "lambda_p:", np.round(srGP.gp_hyper_lambda_p.data.detach().cpu().item(), decimals = 2),
            "sigma_f:", np.round(srGP.gp_hyper_sigma_f.data.detach().cpu().item(), decimals = 2))

In [69]:
print(srGP.gp_hyper_lambda_p.data)

tensor([7.], device='cuda:0')
